In [ ]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_end_of_word = False
        self.frequency = 0

class PrefixTrie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word: str):
        node = self.root
        for char in word:
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
            node.frequency += 1
        node.is_end_of_word = True

    def stem(self, word: str) -> tuple[str, str]:
        if not word: return "", ""
        node = self.root
        path = [node]
        for char in word:
            if char not in node.children: return word, ""
            node = node.children[char]
            path.append(node)

        max_score, split_index = -1, len(word)
        for i, node in enumerate(path[:-1]):
            score = node.frequency * len(node.children)
            if i > 0 and score > max_score:
                max_score, split_index = score, i
        return word[:split_index], word[split_index:]

class SuffixTrie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word: str):
        reversed_word = word[::-1]
        node = self.root
        for char in reversed_word:
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
            node.frequency += 1
        node.is_end_of_word = True

    def stem(self, word: str) -> tuple[str, str]:
        if not word: return "", ""
        reversed_word = word[::-1]
        node = self.root
        path = [node]
        for char in reversed_word:
            if char not in node.children: return word, ""
            node = node.children[char]
            path.append(node)

        max_score, split_index = -1, 0
        for i, current_node in enumerate(path[1:]):
            score = current_node.frequency * len(current_node.children)
            if score > max_score:
                max_score, split_index = score, i + 1

        reversed_suffix = reversed_word[:split_index]
        reversed_stem = reversed_word[split_index:]
        return reversed_stem[::-1], reversed_suffix[::-1]

def read_words_from_file(filename: str) -> list[str]:
    try:
        with open(filename, "r", encoding='utf-8') as f:
            words = [line.strip().lower() for line in f if line.strip()]
        print(f"Successfully loaded {len(words)} words from '{filename}'.")
        return words
    except FileNotFoundError:
        print(f"Error: The file '{filename}' was not found.")
        print("Please make sure you have uploaded the file to the Colab session using the folder icon on the left.")
        return []
    except Exception as e:
        print(f"An error occurred: {e}")
        return []

if __name__ == "__main__":
    file_path = "/content/brown_nouns.txt"

    word_list = read_words_from_file(file_path)

    if word_list:
        print("Building Prefix and Suffix tries...")
        prefix_trie = PrefixTrie()
        suffix_trie = SuffixTrie()

        for word in word_list:
            prefix_trie.insert(word)
            suffix_trie.insert(word)
        print("Tries built successfully.")



        print("\n--- Analysis of Stemming Performance ---")
        for i , word in enumerate(word_list):
            if i > 400 :
              break;
            p_stem, p_suffix = prefix_trie.stem(word)
            print("Prefix Trie Result:")
            print(f"{word} = {p_stem} + {p_suffix if p_suffix else '(no suffix found)'}")

            s_stem, s_suffix = suffix_trie.stem(word)
            print("\nSuffix Trie Result:")
            print(f"{word} = {s_stem} + {s_suffix if s_suffix else '(no suffix found)'}")
            print("-" * 25)

Successfully loaded 202793 words from '/content/brown_nouns.txt'.
Building Prefix and Suffix tries...
Tries built successfully.

--- Analysis of Stemming Performance ---
Prefix Trie Result:
investigation = in + vestigation

Suffix Trie Result:
investigation = investigatio + n
-------------------------
Prefix Trie Result:
primary = p + rimary

Suffix Trie Result:
primary = primar + y
-------------------------
Prefix Trie Result:
election = e + lection

Suffix Trie Result:
election = electio + n
-------------------------
Prefix Trie Result:
evidence = e + vidence

Suffix Trie Result:
evidence = evidenc + e
-------------------------
Prefix Trie Result:
irregularities = i + rregularities

Suffix Trie Result:
irregularities = irregularitie + s
-------------------------
Prefix Trie Result:
place = p + lace

Suffix Trie Result:
place = plac + e
-------------------------
Prefix Trie Result:
jury = j + ury

Suffix Trie Result:
jury = jur + y
-------------------------
Prefix Trie Result:
present

In [ ]:
# Question-02

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from datasets import load_dataset
import re

# font_path = 'path/to/your/NotoSansGujarati-Regular.ttf'

# try:
#     font_prop = fm.FontProperties(fname=font_path)
# except FileNotFoundError:
#     print(f"Font file not found at {font_path}. Gujarati characters may not display correctly.")
#     # Fallback to a generic font family
#     plt.rcParams['font.family'] = 'sans-serif'
#     font_prop = None

dataset = load_dataset("ai4bharat/IndicCorpV2", "indiccorp_v2", streaming=True, split="guj_Gujr")

sample_size = 10000
dataset_sample = list(dataset.take(sample_size))
texts = [item['text'] for item in dataset_sample]

# Tokenize the text
def tokenize(text):
    # Remove punctuation and split by spaces
    text = re.sub(r'[^\w\s]', '', text)
    return text.split()

all_tokens = []
for text in texts:
    all_tokens.extend(tokenize(text))

# Create frequency distribution
freq_dist = {}
for token in all_tokens:
    if token in freq_dist:
        freq_dist[token] += 1
    else:
        freq_dist[token] = 1

# Sort the frequency distribution
sorted_freq_dist = sorted(freq_dist.items(), key=lambda x: x[1], reverse=True)

# Get the top 100 words and their frequencies
top_100_words = sorted_freq_dist[:100]
words = [item[0] for item in top_100_words]
frequencies = [item[1] for item in top_100_words]

# Plot the top 100 most frequent words
plt.figure(figsize=(20, 10))
plt.bar(words, frequencies)
plt.xlabel("Words")
plt.ylabel("Frequency")
plt.title("Top 100 Most Frequent Words")
plt.xticks(rotation=90)
plt.savefig("top_100_words.png")
plt.close()


# Function to remove stop words based on a frequency threshold
def remove_stopwords(freq_dist, threshold):
    return {word: freq for word, freq in freq_dist.items() if freq < threshold}

# Function to plot top 100 words
def plot_top_100(freq_dist, title, filename):
    sorted_freq_dist = sorted(freq_dist.items(), key=lambda x: x[1], reverse=True)
    top_100_words = sorted_freq_dist[:100]
    words = [item[0] for item in top_100_words]
    frequencies = [item[1] for item in top_100_words]

    plt.figure(figsize=(20, 10))
    plt.bar(words, frequencies)
    plt.xlabel("Words")
    plt.ylabel("Frequency")
    plt.title(title)
    plt.xticks(rotation=90)
    plt.savefig(filename)
    plt.close()


# --- Threshold 1 ---
threshold1 = 100

freq_dist_threshold1 = remove_stopwords(freq_dist, threshold1)
print(freq_dist_threshold1)

for word, freq in freq_dist_threshold1.items():
  if freq > 100 :
    print(word)

plot_top_100(freq_dist_threshold1, f"Top 100 Words (Frequency < {threshold1})", "top_100_threshold1.png")


# --- Threshold 2 ---
threshold2 = 500
freq_dist_threshold2 = remove_stopwords(freq_dist, threshold2)
plot_top_100(freq_dist_threshold2, f"Top 100 Words (Frequency < {threshold2})", "top_100_threshold2.png")


# --- Threshold 3 ---
threshold3 = 200
freq_dist_threshold3 = remove_stopwords(freq_dist, threshold3)
plot_top_100(freq_dist_threshold3, f"Top 100 Words (Frequency < {threshold3})", "top_100_threshold3.png")

print("Frequency distribution plots have been generated.")

/tmp/ipython-input-3298030992.py:58: UserWarning: Glyph 2715 (\N{GUJARATI LETTER CHA}) missing from font(s) DejaVu Sans.
  plt.savefig("top_100_words.png")
/tmp/ipython-input-3298030992.py:58: UserWarning: Matplotlib currently does not support Gujarati natively.
  plt.savefig("top_100_words.png")
/tmp/ipython-input-3298030992.py:58: UserWarning: Glyph 2693 (\N{GUJARATI LETTER A}) missing from font(s) DejaVu Sans.
  plt.savefig("top_100_words.png")
/tmp/ipython-input-3298030992.py:58: UserWarning: Glyph 2728 (\N{GUJARATI LETTER NA}) missing from font(s) DejaVu Sans.
  plt.savefig("top_100_words.png")
/tmp/ipython-input-3298030992.py:58: UserWarning: Glyph 2745 (\N{GUJARATI LETTER HA}) missing from font(s) DejaVu Sans.
  plt.savefig("top_100_words.png")
/tmp/ipython-input-3298030992.py:58: UserWarning: Glyph 2724 (\N{GUJARATI LETTER TA}) missing from font(s) DejaVu Sans.
  plt.savefig("top_100_words.png")
/tmp/ipython-input-3298030992.py:58: UserWarning: Glyph 2694 (\N{GUJARATI LETTER AA

{'જઓ': 65, 'ઊઝ': 1, 'મરકટયરડ': 1, 'આજથ': 20, '25': 30, 'જલઈ': 14, 'મથનલ': 2, 'કયથ': 6, 'આખર': 25, 'રજયમ': 86, 'મળલ': 30, 'હર': 40, 'કગરસ': 95, 'અધયકષ': 45, 'ગધ': 39, 'પરતકરય': 18, 'તરપર': 4, 'નગલનડ': 1, 'મઘલયમ': 1, 'જનદશન': 1, 'સવગત': 21, 'કરએ': 58, 'કષતરન': 17, 'વશવસ': 37, 'ફરથ': 45, 'જતવ': 13, 'પરતબદધ': 2, 'આકડ': 59, 'વજન': 32, 'ઘટડવ': 10, 'પરકશનન': 3, 'વતવવ': 2, 'ઉદહરણ': 42, 'અઠવડયમ': 14, 'વકલપ': 41, 'પસદ': 71, 'અગવડતન': 1, 'લકપરય': 31, 'કફર': 3, 'અનલડ': 1, 'ઠકઓ': 1, 'પરથ': 58, 'લમડ': 1, 'ઝલદન': 1, 'બટલગર': 6, 'વદશ': 44, 'દરન': 37, 'મટપય': 5, 'જથથ': 27, 'ખરદ': 60, 'મહસગર': 3, 'હલલ': 5, 'ગધર': 7, 'આણદ': 12, 'નડયદ': 2, 'વડદર': 32, 'શહરજલલમ': 1, 'ઠલવ': 2, 'આબથ': 1, 'સચર': 10, 'પલનપર': 8, 'મહસણમ': 5, 'રટ': 32, 'વરસહન': 1, 'ઉતરવમ': 5, 'જયથ': 5, 'ઉતતર': 46, 'ગજરતમ': 93, 'સપલઈ': 2, 'રજસથનન': 14, 'બચછવડથ': 1, 'શમળજ': 4, 'બરડર': 12, 'હમતનગર': 9, 'ગધનગર': 36, 'અમદવદમ': 60, 'સનલ': 18, 'વનદ': 9, 'દલપ': 8, 'રબર': 7, 'ઠલવય': 2, 'કનદરય': 37, 'જયતરદતય': 2, 'સધયન': 1, 'આશરવદ': 12, 'યતર': 25, 'ઈનદરન'

/tmp/ipython-input-3298030992.py:79: UserWarning: Glyph 2709 (\N{GUJARATI LETTER KA}) missing from font(s) DejaVu Sans.
  plt.savefig(filename)
/tmp/ipython-input-3298030992.py:79: UserWarning: Matplotlib currently does not support Gujarati natively.
  plt.savefig(filename)
/tmp/ipython-input-3298030992.py:79: UserWarning: Glyph 2730 (\N{GUJARATI LETTER PA}) missing from font(s) DejaVu Sans.
  plt.savefig(filename)
/tmp/ipython-input-3298030992.py:79: UserWarning: Glyph 2728 (\N{GUJARATI LETTER NA}) missing from font(s) DejaVu Sans.
  plt.savefig(filename)
/tmp/ipython-input-3298030992.py:79: UserWarning: Glyph 2744 (\N{GUJARATI LETTER SA}) missing from font(s) DejaVu Sans.
  plt.savefig(filename)
/tmp/ipython-input-3298030992.py:79: UserWarning: Glyph 2693 (\N{GUJARATI LETTER A}) missing from font(s) DejaVu Sans.
  plt.savefig(filename)
/tmp/ipython-input-3298030992.py:79: UserWarning: Glyph 2724 (\N{GUJARATI LETTER TA}) missing from font(s) DejaVu Sans.
  plt.savefig(filename)
/tmp/i

Frequency distribution plots have been generated.
